In [0]:
### Build End-to-End Pipeline Dataset: 
### Orders data with: 1)NULL values 2)String salary 3)Date column 
### Tasks: 1)Clean NULLs 2)Cast columns 3)Add derived columns 4)Apply UDF 
### Load: 1)Full load 2)Incremental load 3)Parameterize notebook

In [0]:
### 1. SAMPLE DATASET (Dirty + Realistic)
### 1)NULLs 2)String salary 3)Dates 4)Updates (for incremental load)
### from pyspark.sql import SparkSession

### spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

### data = [
###     (1, "C001", "Laptop", "50000", "2024-01-01"),
###    (2, "C002", "Mobile", None, "2024-01-02"),
###    (3, "C003", "Tablet", "20000", "2024-01-03"),
###    (4, "C004", "Laptop", "55000", "2024-01-04"),
###    (5, "C005", "Headphones", None, "2024-01-05"),
###    ## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
###    (6, "C006", "Camera", "30000", "2024-01-06"),
###    (7, "C007", "Mobile", "18000", "2024-01-07"),
###    (8, "C008", "Watch", "8000", "2024-01-07")
###]

### columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

### df = spark.createDataFrame(data, columns)
### TASK LIST
### Task 1: Handle NULLs
### Replace NULL amount with 1000
### Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date
### Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low
### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High
### Task 5: Full Load
### Load all data to target
### Task 6: Incremental Load
### Load only new/updated records
### Handle duplicates (keep latest)
### Task 7: Parameterization
### Pass:
### input path
### last_loaded_date

In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

In [0]:
data = [
         (1, "C001", "Laptop", "50000", "2024-01-01"),
    (2, "C002", "Mobile", None, "2024-01-02"),
    (3, "C003", "Tablet", "20000", "2024-01-03"),
    (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    ## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
    (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
    (8, "C008", "Watch", "8000", "2024-01-07")
]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)

In [0]:
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


### Type Casting

In [0]:
from pyspark.sql.functions import col, lit, when
df = df.withColumn("amount", col("amount").cast("int"))\
    .withColumn("updated_date", col("updated_date").cast("date"))\
    .withColumn("order_id", col("order_id").cast("int"))
display(df)

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


### Handling NULL Values


In [0]:
df = df.fillna({"amount": 1000})

### Renaming Columns

In [0]:
df = df.withColumnRenamed("updated_date","date")
df.display()

order_id,customer_id,product,amount,date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


### Add Derived Columns

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("bonus", col("amount") * 0.10)

display(df)

order_id,customer_id,product,amount,date,bonus
1,C001,Laptop,50000,2024-01-01,5000.0
2,C002,Mobile,1000,2024-01-02,100.0
3,C003,Tablet,20000,2024-01-03,2000.0
4,C004,Laptop,55000,2024-01-04,5500.0
5,C005,Headphones,1000,2024-01-05,100.0
6,C006,Camera,30000,2024-01-06,3000.0
7,C007,Mobile,18000,2024-01-07,1800.0
8,C008,Watch,8000,2024-01-07,800.0


### Category:
### 20000 → High
### else → Low

In [0]:
df = df.withColumn(
    "Category",
    when(col("amount") >= 20000, "High").otherwise("Low")
)

display(df)

order_id,customer_id,product,amount,date,bonus,Category
1,C001,Laptop,50000,2024-01-01,5000.0,High
2,C002,Mobile,1000,2024-01-02,100.0,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High
4,C004,Laptop,55000,2024-01-04,5500.0,High
5,C005,Headphones,1000,2024-01-05,100.0,Low
6,C006,Camera,30000,2024-01-06,3000.0,High
7,C007,Mobile,18000,2024-01-07,1800.0,Low
8,C008,Watch,8000,2024-01-07,800.0,Low


### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# Step 1: define python function
def amount_bucket_func(amount):
    if amount < 10000:
        return "Low"
    elif amount <= 30000:
        return "Medium"
    else:
        return "High"

# Step 2: convert to UDF
amount_bucket_udf = udf(amount_bucket_func, StringType())

# Step 3: apply on dataframe
df = df.withColumn("amount_bucket", amount_bucket_udf(col("amount")))

display(df)

order_id,customer_id,product,amount,date,bonus,Category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


### Task 5: Full Load
### Load all data to target

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("target_table")

In [0]:
df_check = spark.read.table("target_table")

display(df_check)

order_id,customer_id,product,amount,date,bonus,Category,amount_bucket
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
6,C006,Camera,30000,2024-01-06,3000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,Low,Medium
8,C008,Watch,8000,2024-01-07,800.0,Low,Low


### Remove duplicates (keep latest record)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("order_id").orderBy(col("date").desc())

df_dedup = df.withColumn(
    "rn",
    row_number().over(window_spec)
).filter(
    col("rn") == 1
).drop("rn")

### Incremental Load using MERGE into target_table

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "target_table")

delta_table.alias("t").merge(
    df_dedup.alias("s"),
    "t.order_id = s.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(delta_table)